# Over-Under and SMOTE

In [115]:
import pandas as pd

X_train = pd.read_csv('A2_customer_churn_labeled.csv')

In [116]:
X_train.head(5)

,ID,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary,customer_profile,Y
0,0,585,4,0.00,2,0,1,101728.46,This customer is a 44-year-old female from Spa...,0
1,1,743,6,140348.56,2,1,1,163254.39,This customer is a 32-year-old female from Ger...,0
2,2,527,10,136733.23,1,1,1,57589.29,This customer is a 41-year-old female from Ger...,0
3,3,732,6,98792.40,1,1,0,81491.70,This customer is a 45-year-old female from Ger...,1
4,4,641,3,0.00,2,1,0,116466.19,This customer is a 38-year-old female from Fra...,0


In [109]:
unique_counts = X_train['Y'].value_counts()

print(unique_counts)

0    5565
1    1435
Name: Y, dtype: int64


In [44]:
from sklearn.model_selection import train_test_split

train_data = X_train.drop(columns=['Y','customer_profile']).copy()
train_test_data = X_train['Y'].copy()
x_no_customer_profile = X_train.drop(columns=['customer_profile']).copy()

X = train_data
y = train_test_data

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=4)

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_1, y_train_1)

y_pred = logistic_model.predict(X_test_1)

f1 = f1_score(y_test_1, y_pred)

print(f"F1 Score of Logistic Regression: {f1:.2f}")

In [31]:
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_test_1, y_pred)
print(conf_matrix)

[[1663    0]
 [ 437    0]]


You have 1663 true negatives (TN), which means the model correctly predicted 1663 samples as the negative class.
You have 0 true positives (TP), which means the model did not correctly predict any samples as the positive class.
You have 437 false negatives (FN), which means the model incorrectly predicted 437 samples as the negative class when they actually belong to the positive class.
There are no false positives (FP) in your confusion matrix, which means the model did not incorrectly predict any samples as the positive class when they actually belong to the negative class.

# Over-Sampling

In [61]:
import pandas as pd
import numpy as np

minority_class_label = 1

# Separate minority and majority class samples
minority_samples = x_no_customer_profile[x_no_customer_profile['Y'] == 1]
majority_samples = x_no_customer_profile[x_no_customer_profile['Y'] != 1]

# Determine the number of samples needed for balancing
oversampling_ratio = len(majority_samples) // len(minority_samples)

# Perform bootstrapping for the minority class
oversampled_minority = minority_samples.sample(n=len(minority_samples) * (oversampling_ratio), replace=True)

# Combine oversampled minority class with majority class
oversampled_df = pd.concat([majority_samples, oversampled_minority], ignore_index=True)

In [62]:
oversampled_df.shape

(9870, 9)

In [66]:
from sklearn.model_selection import train_test_split

train_data = oversampled_df.drop(columns=['Y']).copy()
train_test_data = oversampled_df['Y'].copy()

X = train_data
y = train_test_data

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=42)

In [67]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_1, y_train_1)

y_pred = logistic_model.predict(X_test_1)

f1 = f1_score(y_test_1, y_pred)

print(f"F1 Score of Logistic Regression: {f1:.2f}")

F1 Score of Logistic Regression: 0.38


In [68]:
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_test_1, y_pred)
print(conf_matrix)

[[1245  435]
 [ 873  408]]


# Under-Sampling

In [70]:
import pandas as pd
import numpy as np

minority_class_label = 1

# Separate minority and majority class samples
minority_samples = x_no_customer_profile[x_no_customer_profile['Y'] == minority_class_label]
majority_samples = x_no_customer_profile[x_no_customer_profile['Y'] != minority_class_label]

# Determine the number of samples needed for balancing
undersampling_ratio = len(minority_samples) / len(majority_samples)

print(undersampling_ratio)

0.2578616352201258


In [71]:
# Perform under-sampling for the majority class
undersampled_majority = majority_samples.sample(frac=undersampling_ratio, random_state=42)

# Combine undersampled majority class with minority class
undersampled_df = pd.concat([minority_samples, undersampled_majority], ignore_index=True)

In [73]:
undersampled_df.shape

(2870, 9)

In [105]:
unique_counts = undersampled_df['Y'].value_counts()

print(unique_counts)

1    1435
0    1435
Name: Y, dtype: int64


In [77]:
from sklearn.model_selection import train_test_split

train_data = undersampled_df.drop(columns=['Y']).copy()
train_test_data = undersampled_df['Y'].copy()

X = train_data
y = train_test_data

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.2, random_state=42)

In [78]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_1, y_train_1)

y_pred = logistic_model.predict(X_test_1)

f1 = f1_score(y_test_1, y_pred)

print(f"F1 Score of Logistic Regression: {f1:.2f}")

F1 Score of Logistic Regression: 0.60


In [79]:
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_test_1, y_pred)
print(conf_matrix)

[[139 127]
 [120 188]]


# SMOTE

The sampling_strategy parameter in the SMOTE algorithm of the imbalanced-learn library allows you to specify how you want to balance the class distribution. Here are some common values you can use for the sampling_strategy parameter:

'auto': This is the default value and is recommended for most cases. It automatically adjusts the sampling strategy to balance the classes by generating synthetic samples for the minority class.

float: You can specify a float value between 0 and 1, which represents the desired ratio of the number of samples in the minority class to the majority class after resampling. For example, if you set sampling_strategy=0.5, it aims to make the minority class size half the size of the majority class.

str: You can specify a string that indicates the desired resampling strategy. Some common options include:
'minority': Resample only the minority class to match the majority class size.
'not minority': Resample all classes except the minority class.
'majority': Resample only the majority class to match the minority class size.
'not majority': Resample all classes except the majority class.

dict: You can provide a dictionary that explicitly specifies the number of samples to be generated for each class. For example, sampling_strategy={0: 1000, 1: 500} means generating 1000 synthetic samples for class 0 and 500 synthetic samples for class 1.

The choice of sampling_strategy depends on your specific problem and the desired balance between classes. 'auto' often works well as it automatically adjusts to create a balanced dataset. However, you may need to experiment with different strategies based on the characteristics of your data and the problem you're solving.

# auto

In [97]:
import pandas as pd
from imblearn.over_sampling import SMOTE

# Assuming you have a DataFrame 'df' with a column 'Y' representing class labels
# and 'minority_class_label' is the label of the minority class (e.g., 1)
minority_class_label = 1

# Separate features (X) and class labels (y)
X = x_no_customer_profile.drop(columns=['Y'])
y = x_no_customer_profile['Y']

# Initialize SMOTE with the desired sampling strategy
smote = SMOTE(sampling_strategy='auto', random_state=42)

# Apply SMOTE to generate synthetic samples
X_resampled, y_resampled = smote.fit_resample(X, y)

In [98]:
unique_counts = y_resampled.value_counts()

print(unique_counts)

0    5565
1    5565
Name: Y, dtype: int64


In [99]:
X_resampled

,ID,credit_score,tenure,balance,number_of_products,has_credit_card,is_active_member,salary
0,0,585,4,0.000000,2,0,1,101728.460000
1,1,743,6,140348.560000,2,1,1,163254.390000
2,2,527,10,136733.230000,1,1,1,57589.290000
3,3,732,6,98792.400000,1,1,0,81491.700000
4,4,641,3,0.000000,2,1,0,116466.190000
...,...,...,...,...,...,...,...,...
11125,4912,660,2,133801.451833,1,1,0,55169.030495
11126,2845,756,4,139700.244658,3,0,0,129781.145544
11127,4966,521,7,74988.974295,2,0,0,103316.631333
11128,4580,702,3,204362.517259,2,0,1,132481.863149


In [100]:
from sklearn.model_selection import train_test_split

train_data = X_resampled.copy()
train_test_data = y_resampled.copy()

X = train_data
y = train_test_data

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=42)

In [101]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_1, y_train_1)

y_pred = logistic_model.predict(X_test_1)

f1 = f1_score(y_test_1, y_pred)

print(f"F1 Score of Logistic Regression: {f1:.2f}")

F1 Score of Logistic Regression: 0.59


In [102]:
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_test_1, y_pred)
print(conf_matrix)

[[1018  666]
 [ 673  982]]


--------------

In [103]:
import pandas as pd
from imblearn.over_sampling import SMOTE

# Assuming you have a DataFrame 'df' with a column 'Y' representing class labels
# and 'minority_class_label' is the label of the minority class (e.g., 1)
minority_class_label = 1

# Separate features (X) and class labels (y)
X = x_no_customer_profile.drop(columns=['Y'])
y = x_no_customer_profile['Y']

# Initialize SMOTE with the desired sampling strategy
smote = SMOTE(sampling_strategy=0.75, random_state=42)

# Apply SMOTE to generate synthetic samples
X_resampled, y_resampled = smote.fit_resample(X, y)

unique_counts = y_resampled.value_counts()

print(unique_counts)

0    5565
1    4173
Name: Y, dtype: int64


In [104]:
from sklearn.model_selection import train_test_split

train_data = X_resampled.copy()
train_test_data = y_resampled.copy()

X = train_data
y = train_test_data

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_1, y_train_1)

y_pred = logistic_model.predict(X_test_1)

f1 = f1_score(y_test_1, y_pred)

print(f"F1 Score of Logistic Regression: {f1:.2f}")

from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_test_1, y_pred)
print(conf_matrix)

F1 Score of Logistic Regression: 0.40
[[1313  362]
 [ 847  400]]


------------

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd

In [113]:
def sampling_methods(X,sampling_method,sampling_ratio = None):
    
    minority_class_label = 1
    
    if sampling_method == 'over':

        minority_samples = X[X['Y'] == minority_class_label]
        majority_samples = X[X['Y'] != minority_class_label]
        
        if sampling_ratio == None:
            ratio = len(majority_samples) // len(minority_samples)
        else:
            ratio = sampling_ratio
        oversampled_minority = minority_samples.sample(n=len(minority_samples) * (ratio), replace=True)

        df = pd.concat([majority_samples, oversampled_minority], ignore_index=True)
    
    elif sampling_method == 'under':

        minority_samples = X[X['Y'] == minority_class_label]
        majority_samples = X[X['Y'] != minority_class_label]

        if sampling_ratio == None:
            ratio = len(minority_samples) / len(majority_samples)
        else:
            ratio = sampling_ratio
        
        undersampled_majority = majority_samples.sample(frac=undersampling_ratio, random_state= 42)

        df = pd.concat([minority_samples, undersampled_majority], ignore_index=True)
    
    elif sampling_method == 'SMOTE':
        
        minority_samples = X[X['Y'] == minority_class_label]
        majority_samples = X[X['Y'] != minority_class_label]

        if sampling_ratio == None:
            ratio = 'auto'
        else:
            ratio = sampling_ratio
        

        features = X.drop(columns=['Y'])
        label = X['Y']
        
        smote = SMOTE(sampling_strategy= ratio, random_state=42)
        X_resampled, y_resampled = smote.fit_resample(features, label)
        
        df = pd.concat([pd.DataFrame(X_resampled, columns=features.columns),
                                  pd.Series(y_resampled, name='Y')], axis=1)

    return df

In [129]:
train_data = X_train.drop(columns=['customer_profile']).copy()
X_resampled = sampling_methods(train_data,'under')

In [130]:
unique_counts = X_resampled['Y'].value_counts()

print(unique_counts)

1    1435
0    1435
Name: Y, dtype: int64


In [131]:
from sklearn.model_selection import train_test_split

train_data = X_resampled.drop(columns=['Y']).copy()
train_test_data = X_resampled['Y'].copy()

X = train_data
y = train_test_data

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=42)

In [132]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logistic_model = LogisticRegression(random_state=42)

logistic_model.fit(X_train_1, y_train_1)

y_pred = logistic_model.predict(X_test_1)

f1 = f1_score(y_test_1, y_pred)

print(f"F1 Score of Logistic Regression: {f1:.2f}")

F1 Score of Logistic Regression: 0.59
